# 🦺 SafeWalk Navigation — 8클래스 YOLO 학습 파이프라인 v8

## v8 개선 사항
- **모델**: YOLOv8x (최대 크기, 최고 정확도)
- **데이터**: Roboflow 전체 + COCO + people 전용 데이터 2개 추가
- **학습**: Phase 1 (클래스별 개별) → Phase 2 (통합 파인튜닝)
- **출력**: 한글 완전 지원, 에폭별 상세 로그, 입문자 친화적 결과
- **실시간**: TensorRT FP16 export (132 FPS)

## 클래스 목록
| ID | 클래스 | 설명 |
|----|--------|------|
| 0 | bicycle | 자전거 |
| 1 | kickboard | 킥보드 (전동 킥보드 포함) |
| 2 | bollard | 볼라드 (도로 분리대) |
| 3 | car | 자동차 |
| 4 | motorcycle | 오토바이 |
| 5 | stairs | 계단 / 에스컬레이터 |
| 6 | crosswalk | 횡단보도 |
| 7 | people | 보행자 |

## v7 → v8 성능 목표
| 클래스 | v7 결과 | v8 목표 |
|--------|---------|---------|
| bicycle | 62.2% | 75%+ |
| kickboard | 54.2% | 70%+ |
| bollard | 74.2% | 83%+ |
| car | 79.8% | 85%+ |
| motorcycle | 72.6% | 82%+ |
| stairs | 67.9% | 82%+ |
| crosswalk | 76.6% | 85%+ |
| people | 34.5% | 60%+ |

## Cell 0 — 패키지 설치 & 환경 설정

In [ ]:
# ============================================================
# Cell 0: 패키지 설치 & 환경 설정
# ============================================================
import subprocess, sys, os

print("=" * 60)
print(" 패키지 설치 중...")
print("=" * 60)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "ultralytics>=8.3.0", "roboflow",
    "opencv-python-headless", "matplotlib",
    "pandas", "pyyaml", "seaborn",
    "--upgrade"
], check=True)

# nvrtc 심링크 자동 수정
nvrtc_src = "/usr/local/lib/python3.10/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0"
nvrtc_dst = "/usr/local/lib/libnvrtc-builtins.so.13.0"
if os.path.exists(nvrtc_src) and not os.path.exists(nvrtc_dst):
    subprocess.run(["ln", "-sf", nvrtc_src, nvrtc_dst], check=True)
    subprocess.run(["ldconfig"], check=True)
    print("✅ nvrtc 심링크 설정 완료")

import shutil, random, yaml, time, json
from pathlib import Path
from collections import defaultdict

import cv2, numpy as np, torch
from ultralytics import YOLO

# ---- GPU 확인 ----
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM  = torch.cuda.get_device_properties(0).total_memory / 1e9
    DEVICE   = "0"
    print(f"✅ GPU 감지: {GPU_NAME} ({GPU_MEM:.1f} GB)")
else:
    DEVICE = "cpu"
    print("⚠️ CPU 모드 (GPU 없음 — 학습 매우 느림)")

# ---- 배치 크기 자동 설정 (YOLOv8x 기준) ----
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 32 if vram_gb >= 24 else 16 if vram_gb >= 16 else 8 if vram_gb >= 10 else 4
    print(f"   VRAM {vram_gb:.1f}GB → 배치 크기: {BATCH}")
else:
    BATCH = 2

# ---- 폴더 구조 ----
PROJECT_ROOT  = Path("/root/pbl2-main")
DATA_DIR      = PROJECT_ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
MERGED_DIR    = DATA_DIR / "merged"
PROC_DIR      = DATA_DIR / "processed"
CONFIG_DIR    = PROJECT_ROOT / "configs"
MODELS_DIR    = PROJECT_ROOT / "models"
PHASE1_DIR    = MODELS_DIR / "phase1_per_class"
PHASE2_DIR    = MODELS_DIR / "safewalk_v8"

for d in [RAW_DIR, MERGED_DIR, PROC_DIR, CONFIG_DIR, MODELS_DIR, PHASE1_DIR, PHASE2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CONFIG_DIR / "dataset_8class.yaml"

CLASS_NAMES = ["bicycle", "kickboard", "bollard", "car",
               "motorcycle", "stairs", "crosswalk", "people"]
CLASS_KO    = ["자전거", "킥보드", "볼라드", "자동차",
               "오토바이", "계단", "횡단보도", "보행자"]
NUM_CLASSES = len(CLASS_NAMES)
RF_API_KEY  = os.environ["ROBOFLOW_API_KEY"]

print()
print("=" * 60)
print(" SafeWalk v8 환경 설정 완료!")
print("=" * 60)
print(f" 프로젝트 경로 : {PROJECT_ROOT}")
print(f" GPU          : {GPU_NAME if torch.cuda.is_available() else 'CPU'}")
print(f" 배치 크기    : {BATCH}")
print(f" 클래스 수    : {NUM_CLASSES}개")
print("=" * 60)

## Cell 1 — 데이터셋 다운로드 (Roboflow + COCO)

In [ ]:
# ============================================================
# Cell 1: 데이터셋 다운로드
# ============================================================
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)

RF_DATASETS = [
    # ===== 복합 데이터셋 =====
    {
        "workspace": "esera", "project": "bollards-crosswalk-stairs", "version": 1,
        "class_map": {
            "bollard": 2, "bollards": 2, "traffic-cone": 2,
            "stairs": 5, "stair": 5, "step": 5, "steps": 5, "staircase": 5, "escalator": 5,
            "crosswalk": 6, "zebra-crossing": 6, "pedestrian-crossing": 6,
        },
        "out_name": "bollard_stairs_crosswalk",
    },
    {
        "workspace": "engs-89", "project": "bikes-pedestrians-scooters-wheelchairs", "version": 1,
        "class_map": {
            "scooter": 1, "e-scooter": 1, "electric-scooter": 1, "kickboard": 1, "kick-scooter": 1,
            "bicycle": 0, "bike": 0,
            "person": 7, "pedestrian": 7, "human": 7,
        },
        "out_name": "kickboard_bicycle_people",
    },
    # ===== crosswalk =====
    {"workspace": "ryu-evgdg",          "project": "crosswalk-gybkt",  "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6, "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_new_1"},
    {"workspace": "edgar1019-naver-com", "project": "crosswalk-ognxu",  "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6, "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_new_2"},
    {"workspace": "tcc-xn3st",          "project": "crosswalk-djojj",  "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6, "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_new_3"},
    {"workspace": "wqwdas",             "project": "crosswalk-1elwe",  "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6, "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_1"},
    {"workspace": "kmong",              "project": "crosswalk-i59cr",  "version": 1,
     "class_map": {"crosswalk": 6, "cross-walk": 6, "zebra": 6, "zebra-crossing": 6, "pedestrian-crossing": 6, "walk": 6},
     "out_name": "crosswalk_2"},
    # ===== bollard =====
    {"workspace": "facility",   "project": "bollard-3icj2", "version": 1,
     "class_map": {"bollard": 2, "bollards": 2, "traffic-cone": 2, "cone": 2, "post": 2},
     "out_name": "bollard_1"},
    {"workspace": "shin-7ty4z", "project": "bollard-ds7rv", "version": 1,
     "class_map": {"bollard": 2, "bollards": 2, "traffic-cone": 2, "cone": 2, "post": 2},
     "out_name": "bollard_2"},
    # ✨ v8.1 신규 볼라드 추가 (기존 549장 → 약 1,700장)
    {"workspace": "project-m4zhi",  "project": "bollard-8pt2h",                    "version": 3,
     "class_map": {"bollard": 2, "bollards": 2, "cone": 2, "post": 2},
     "out_name": "bollard_new_1"},
    {"workspace": "project-60htx",  "project": "bollard-v2gn5",                    "version": 1,
     "class_map": {"bollard": 2, "bollards": 2, "bollard_normal": 2, "bollard_abnormal": 2,
                   "tubular-marker-normal": 2, "tubular-marker-abnormal": 2},
     "out_name": "bollard_new_2"},
    {"workspace": "testing-0bbg4",  "project": "bollard_port_of_antwerpen_public", "version": 1,
     "class_map": {"bollard": 2, "bollards": 2},
     "out_name": "bollard_new_3"},

    # ===== motorcycle =====
    {"workspace": "bollard",    "project": "motorcycle-l9wuv", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_1"},
    {"workspace": "aflakha",    "project": "motorcycle-illvg", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_2"},
    {"workspace": "sagemcom-j0etp", "project": "motorcycle-m1pbt", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_3"},
    {"workspace": "free-brb5x", "project": "motorcycle-wcwl7", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_new_1"},
    {"workspace": "yolov11da",  "project": "motorcycle-v3pri", "version": 1,
     "class_map": {"motorcycle": 4, "motorbike": 4, "moto": 4, "bike": 4},
     "out_name": "motorcycle_new_2"},
    # ===== stairs =====
    {"workspace": "stairs-n9kkx",           "project": "stairs-lusiz",    "version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5},
     "out_name": "stairs_1"},
    {"workspace": "escalatorstairsdetection","project": "escalator-stairs","version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5, "escalators": 5},
     "out_name": "stairs_2"},
    {"workspace": "airport-vmrjj", "project": "stairs-zns8e", "version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5},
     "out_name": "stairs_new_1"},
    {"workspace": "pitom",        "project": "stairs-2igna", "version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5},
     "out_name": "stairs_new_2"},
    {"workspace": "saffpaff",     "project": "my-stairs",    "version": 1,
     "class_map": {"stairs": 5, "stair": 5, "step": 5, "staircase": 5, "escalator": 5},
     "out_name": "stairs_new_3"},
    # ===== bicycle =====
    {"workspace": "thesis-2-eoeid","project": "bicycle-3ln52", "version": 1,
     "class_map": {"bicycle": 0, "bike": 0, "cycle": 0},
     "out_name": "bicycle_new_1"},
    {"workspace": "it-vedant",     "project": "bicycle-cocvv", "version": 1,
     "class_map": {"bicycle": 0, "bike": 0, "cycle": 0},
     "out_name": "bicycle_new_2"},
    {"workspace": "scgs-ygroy",    "project": "bicycle-qwalo", "version": 1,
     "class_map": {"bicycle": 0, "bike": 0, "cycle": 0},
     "out_name": "bicycle_new_3"},
    # ===== kickboard =====
    {"workspace": "vkie39",        "project": "kickboard-j1mfx", "version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1, "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_1"},
    {"workspace": "inha-univ-vgzgz","project": "kickboard-ibhkj","version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1, "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_2"},
    {"workspace": "school-zr91f",  "project": "kickboard-er5c4","version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1, "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_3"},
    {"workspace": "dklyolo1",      "project": "kickboard-dfkjo","version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1, "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_4"},
    {"workspace": "cjs2778",       "project": "kickboard-fdiqz","version": 1,
     "class_map": {"kickboard": 1, "kick-board": 1, "scooter": 1, "e-scooter": 1, "kick-scooter": 1, "personal-mobility": 1},
     "out_name": "kickboard_new_1"},
    # ===== people =====
    {"workspace": "project-fy",              "project": "pedestrian-detection-gpggm","version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_1"},
    {"workspace": "samproject1",             "project": "pedestrian-detection-vsmzo","version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_2"},
    {"workspace": "pedestriantracking-0t91d","project": "pedestrian-detection-fqlmr","version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_3"},
    # v8 신규 추가
    {"workspace": "vk-featureextractor",        "project": "person-detection-9a6mk",     "version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_new_1"},
    {"workspace": "computer-vision-project-mjtmc","project": "people-detection-general",  "version": 1,
     "class_map": {"person": 7, "people": 7, "pedestrian": 7, "human": 7, "man": 7, "woman": 7},
     "out_name": "people_new_2"},
]


def parse_yaml_names(yaml_path):
    try:
        with open(yaml_path, encoding="utf-8") as f:
            dy = yaml.safe_load(f)
        raw = dy.get("names", [])
        if isinstance(raw, list):
            return {i: n for i, n in enumerate(raw)}
        elif isinstance(raw, dict):
            return {int(k): str(v) for k, v in raw.items()}
    except:
        pass
    return {}


def build_id_map(orig_names, class_map):
    id_map = {}
    for orig_id, orig_name in orig_names.items():
        for alias, our_id in class_map.items():
            if str(orig_name).lower().strip() == alias.lower().strip():
                id_map[int(orig_id)] = our_id
                break
    if not id_map and orig_names:
        for oid, nid in zip(sorted(orig_names), sorted(set(class_map.values()))):
            id_map[oid] = nid
    return id_map


def remap_and_copy(src_root, class_map, out_name):
    out_img = RAW_DIR / out_name / "images"
    out_lbl = RAW_DIR / out_name / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    yaml_path = src_root / "data.yaml"
    if not yaml_path.exists():
        found = list(src_root.rglob("data.yaml"))
        yaml_path = found[0] if found else None
    orig_names = parse_yaml_names(yaml_path) if yaml_path else {}
    print(f"     원본 클래스: {orig_names}")
    id_map = build_id_map(orig_names, class_map)
    print(f"     ID 매핑: {id_map}")
    if not id_map:
        print(f"     ❌ 매핑 실패 — 스킵")
        return 0
    converted = 0
    for split in ["train", "valid", "test"]:
        lbl_dir = src_root / split / "labels"
        img_dir = src_root / split / "images"
        if not lbl_dir.exists():
            continue
        for lbl_file in lbl_dir.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            new_lines = []
            for line in lines:
                parts = line.split()
                if not parts:
                    continue
                orig_id = int(parts[0])
                if orig_id not in id_map:
                    continue
                parts[0] = str(id_map[orig_id])
                new_lines.append(" ".join(parts))
            if not new_lines:
                continue
            img_src = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                cand = img_dir / (lbl_file.stem + ext)
                if cand.exists():
                    img_src = cand
                    break
            if not img_src:
                continue
            new_stem = f"{out_name}_{split}_{lbl_file.stem}"
            shutil.copy2(img_src, out_img / (new_stem + img_src.suffix))
            (out_lbl / (new_stem + ".txt")).write_text(
                "\n".join(new_lines), encoding="utf-8")
            converted += 1
    print(f"     ✅ 저장: {converted}개")
    return converted


print("=" * 60)
print(" Roboflow 데이터셋 다운로드")
print(f" 총 {len(RF_DATASETS)}개 데이터셋")
print("=" * 60)

download_results = {}
for i, cfg in enumerate(RF_DATASETS, 1):
    out_img = RAW_DIR / cfg["out_name"] / "images"
    if out_img.exists() and len(list(out_img.glob("*.*"))) > 50:
        cnt = len(list(out_img.glob("*.*")))
        print(f"[{i:2d}/{len(RF_DATASETS)}] ⏭️  {cfg['out_name']} ({cnt}장 이미 존재, 스킵)")
        download_results[cfg["out_name"]] = cnt
        continue
    print(f"\n[{i:2d}/{len(RF_DATASETS)}] 다운로드 중: {cfg['workspace']}/{cfg['project']}")
    try:
        proj    = rf.workspace(cfg["workspace"]).project(cfg["project"])
        dataset = proj.version(cfg["version"]).download(
            "yolov8", location=str(RAW_DIR / cfg["project"]), overwrite=False)
        n = remap_and_copy(Path(dataset.location), cfg["class_map"], cfg["out_name"])
        download_results[cfg["out_name"]] = n
        raw_dir = RAW_DIR / cfg["project"]
        if raw_dir.exists():
            shutil.rmtree(raw_dir)
    except Exception as e:
        print(f"     ❌ 실패: {e}")
        download_results[cfg["out_name"]] = 0

# ---- COCO 다운로드 (네트워크 오류 자동 재시도) ----
print("\n" + "=" * 60)
print(" COCO 2017 다운로드 (bicycle / car / motorcycle / person)")
print("=" * 60)
try:
    import fiftyone.zoo as foz
    import time as _time
    COCO_MAP = {
        "bicycle":    (0, 2000),
        "car":        (3, 2000),
        "motorcycle": (4, 2000),
        "person":     (7, 5000),
    }
    for coco_cls, (our_id, max_samples) in COCO_MAP.items():
        out_img = RAW_DIR / f"coco_{coco_cls}" / "images"
        out_lbl = RAW_DIR / f"coco_{coco_cls}" / "labels"
        existing = len(list(out_img.glob("*.*"))) if out_img.exists() else 0
        if existing >= int(max_samples * 0.7):
            print(f"  ⏭️  coco_{coco_cls} 이미 {existing}장 존재, 스킵")
            continue
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)
        print(f"  📥 coco_{coco_cls} 다운로드 중... (최대 {max_samples}장)")

        for attempt in range(3):  # 최대 3번 재시도
            try:
                ds = foz.load_zoo_dataset("coco-2017", split="train",
                    label_types=["detections"], classes=[coco_cls],
                    max_samples=max_samples,
                    dataset_name=f"coco_{coco_cls}_v8", overwrite=True)
                converted = 0
                for sample in ds:
                    if not sample.filepath or not Path(sample.filepath).exists():
                        continue
                    lines = []
                    if sample.ground_truth:
                        for det in sample.ground_truth.detections:
                            if det.label.lower() != coco_cls.lower():
                                continue
                            x1, y1, bw, bh = det.bounding_box
                            if bw <= 0 or bh <= 0:
                                continue
                            cx, cy = x1 + bw / 2, y1 + bh / 2
                            lines.append(f"{our_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                    if not lines:
                        continue
                    src_f = Path(sample.filepath)
                    shutil.copy2(src_f, out_img / src_f.name)
                    (out_lbl / (src_f.stem + ".txt")).write_text("\n".join(lines), encoding="utf-8")
                    converted += 1
                print(f"  ✅ coco_{coco_cls}: {converted}장")
                break
            except Exception as e:
                print(f"  ⚠️  시도 {attempt+1}/3 실패: {e}")
                if attempt < 2:
                    print(f"     5초 후 재시도...")
                    _time.sleep(5)
                else:
                    print(f"  ❌ coco_{coco_cls} 다운로드 포기 — 계속 진행")
except Exception as e:
    print(f"  ❌ COCO 실패: {e}")

# ---- car 데이터 없으면 Roboflow 대체 ----
car_img_dir = RAW_DIR / "coco_car" / "images"
car_count = len(list(car_img_dir.glob("*.*"))) if car_img_dir.exists() else 0
if car_count < 100:
    print("\n⚠️  car COCO 데이터 부족 → Roboflow 대체 다운로드")
    CAR_FALLBACK = [
        {"workspace": "sam-s7wwp",  "project": "car-detection-system", "version": 5,
         "class_map": {"car": 3, "vehicle": 3, "automobile": 3},
         "out_name": "car_fallback_1"},
        {"workspace": "brad-dwyer", "project": "car-vs-truck",         "version": 1,
         "class_map": {"car": 3, "vehicle": 3},
         "out_name": "car_fallback_2"},
    ]
    for cfg in CAR_FALLBACK:
        out_img_c = RAW_DIR / cfg["out_name"] / "images"
        if out_img_c.exists() and len(list(out_img_c.glob("*.*"))) > 50:
            print(f"  ⏭️  {cfg['out_name']} 이미 존재, 스킵")
            continue
        print(f"  📥 {cfg['workspace']}/{cfg['project']} 다운로드 중...")
        try:
            proj    = rf.workspace(cfg["workspace"]).project(cfg["project"])
            dataset = proj.version(cfg["version"]).download(
                "yolov8", location=str(RAW_DIR / cfg["project"]), overwrite=False)
            n = remap_and_copy(Path(dataset.location), cfg["class_map"], cfg["out_name"])
            raw_dir_c = RAW_DIR / cfg["project"]
            if raw_dir_c.exists(): shutil.rmtree(raw_dir_c)
            print(f"  ✅ car 대체 데이터: {n}장")
        except Exception as e:
            print(f"  ❌ 실패: {e}")

# ---- 클래스별 RAW 통계 ----
print("\n" + "=" * 60)
print(" 클래스별 원본 데이터 현황")
print("=" * 60)
raw_cnt = defaultdict(int)
for lbl_dir in RAW_DIR.rglob("labels"):
    for lf in lbl_dir.glob("*.txt"):
        for line in lf.read_text(encoding="utf-8").strip().splitlines():
            parts = line.split()
            if parts:
                try: raw_cnt[int(parts[0])] += 1
                except: pass

total_raw = sum(raw_cnt.values())
print(f"{'클래스':<12} {'어노테이션 수':>12} {'비율':>8} {'상태':>6}")
print("-" * 45)
for cid, cname in enumerate(CLASS_NAMES):
    cnt  = raw_cnt.get(cid, 0)
    pct  = cnt / total_raw * 100 if total_raw else 0
    flag = "❌ 부족" if cnt < 500 else ("⚠️  적음" if cnt < 2000 else "✅ 충분")
    print(f"[{cid}] {CLASS_KO[cid]:<10} {cnt:>10,}개 {pct:>7.1f}% {flag}")
print(f"{'합계':<12} {total_raw:>12,}개")

## Cell 2 — 데이터 병합 & 클래스 균형화

In [ ]:
# ============================================================
# Cell 2: 병합 & 클래스 불균형 해소
# v8.1 수정:
#   - 세그멘테이션 라벨 자동 필터링 (보행자 문제 해결)
#   - 볼라드 최소 테스트셋 보장 (MIN_TEST_PER_CLASS)
# ============================================================
print("=" * 60)
print(" 데이터 병합 시작")
print("=" * 60)

MERGED_IMAGES = MERGED_DIR / "images"
MERGED_LABELS = MERGED_DIR / "labels"

if MERGED_IMAGES.exists(): shutil.rmtree(MERGED_IMAGES)
if MERGED_LABELS.exists(): shutil.rmtree(MERGED_LABELS)
MERGED_IMAGES.mkdir(parents=True, exist_ok=True)
MERGED_LABELS.mkdir(parents=True, exist_ok=True)


def is_bbox_line(line):
    """세그멘테이션 라벨 걸러내기: YOLO bbox는 정확히 5개 숫자"""
    parts = line.strip().split()
    return len(parts) == 5


def merge_raw_folder(folder_name):
    base = RAW_DIR / folder_name
    candidates = [base]
    for split in ["train", "valid", "val", "test"]:
        if (base / split).exists():
            candidates.append(base / split)
    copied = seg_removed = 0
    for src_root in candidates:
        src_img = src_root / "images"
        src_lbl = src_root / "labels"
        if not src_lbl.exists():
            continue
        for lbl_file in src_lbl.glob("*.txt"):
            lines = lbl_file.read_text(encoding="utf-8").strip().splitlines()
            # 세그멘테이션 라인 제거 (5개 숫자가 아닌 줄)
            bbox_lines = [l for l in lines if is_bbox_line(l)]
            seg_cnt = len(lines) - len(bbox_lines)
            if seg_cnt > 0:
                seg_removed += seg_cnt
            if not bbox_lines:
                continue
            new_stem = f"{folder_name}_{src_root.name}_{lbl_file.stem}"
            (MERGED_LABELS / (new_stem + ".txt")).write_text(
                "\n".join(bbox_lines), encoding="utf-8")
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                img_src = src_img / (lbl_file.stem + ext)
                if img_src.exists():
                    shutil.copy2(img_src, MERGED_IMAGES / (new_stem + ext))
                    break
            copied += 1
    return copied, seg_removed


total_merged = 0
total_seg_removed = 0
folders = sorted(d.name for d in RAW_DIR.iterdir() if d.is_dir())
for i, folder in enumerate(folders, 1):
    n, seg = merge_raw_folder(folder)
    status = "✅" if n > 0 else "❌"
    seg_str = f"  (세그 {seg}개 제거)" if seg > 0 else ""
    print(f"  [{i:2d}/{len(folders)}] {status} {folder:<40}: {n:>5}개{seg_str}")
    total_merged += n
    total_seg_removed += seg

print(f"\n  📦 총 병합: {total_merged:,}개")
if total_seg_removed > 0:
    print(f"  🧹 세그멘테이션 라벨 제거: {total_seg_removed:,}개 (보행자 품질 개선)")

# ---- 병합 전 분포 확인 ----
class_counter = defaultdict(int)
for lf in MERGED_LABELS.glob("*.txt"):
    for line in lf.read_text(encoding="utf-8").strip().splitlines():
        parts = line.split()
        if parts:
            try: class_counter[int(parts[0])] += 1
            except: pass

print("\n[캡 적용 전 클래스별 분포]")
max_cnt = max(class_counter.values()) if class_counter else 1
for cid, cname in enumerate(CLASS_NAMES):
    cnt = class_counter.get(cid, 0)
    bar = "█" * int(cnt / max_cnt * 35)
    print(f"  [{cid}] {CLASS_KO[cid]:<8} {cnt:>8,}개  {bar}")

# ---- 클래스 캡 설정 ----
MAX_PER_CLASS = {
    0: 12000,  # bicycle
    1: 15000,  # kickboard
    2: 7000,   # bollard
    3: 5000,   # car
    4: 7000,   # motorcycle
    5: 8000,   # stairs
    6: 99999,  # crosswalk
    7: 15000,  # people
}

print("\n[클래스 캡 적용 중...]")
for cid, cname in enumerate(CLASS_NAMES):
    cap = MAX_PER_CLASS.get(cid, 99999)
    cap_str = f"{cap:,}개" if cap < 99999 else "제한 없음"
    print(f"  [{cid}] {CLASS_KO[cid]:<8} {cap_str:>10}")

random.seed(42)
all_lbls = list(MERGED_LABELS.glob("*.txt"))
random.shuffle(all_lbls)

live = defaultdict(int)
kept = removed = 0

for lbl in all_lbls:
    lines = lbl.read_text(encoding="utf-8").strip().splitlines()
    classes = []
    for l in lines:
        parts = l.split()
        if parts:
            try: classes.append(int(parts[0]))
            except: pass
    if not classes:
        lbl.unlink()
        removed += 1
        continue
    main_cls = max(set(classes), key=classes.count)
    if live[main_cls] >= MAX_PER_CLASS.get(main_cls, 99999):
        for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
            img = MERGED_IMAGES / (lbl.stem + ext)
            if img.exists():
                img.unlink()
                break
        lbl.unlink()
        removed += 1
    else:
        for c in classes:
            live[c] += 1
        kept += 1

print(f"\n  ✅ 유지: {kept:,}개  ❌ 제거: {removed:,}개")
print("\n[캡 적용 후 클래스별 분포]")
for cid, cname in enumerate(CLASS_NAMES):
    cnt  = live.get(cid, 0)
    bar  = "█" * min(cnt // 400, 35)
    flag = "⚠️  부족!" if cnt < 500 else "✅"
    print(f"  [{cid}] {CLASS_KO[cid]:<8} {cnt:>8,}개  {bar} {flag}")

## Cell 3 — Train/Val/Test 분할 & YAML 생성

In [ ]:
# ============================================================
# Cell 3: 데이터 분할 (7:2:1) + 클래스별 서브셋 생성
# v8.1 수정:
#   - 볼라드 등 소수 클래스 최소 테스트셋 보장
#   - MIN_TEST_PER_CLASS: 클래스별 최소 테스트 이미지 수
# ============================================================
print("=" * 60)
print(" 데이터 분할 (Train 70% / Val 20% / Test 10%)")
print("=" * 60)

if PROC_DIR.exists(): shutil.rmtree(PROC_DIR)
for split in ["train", "val", "test"]:
    (PROC_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PROC_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

all_images = sorted([f for f in MERGED_IMAGES.glob("*.*")
                     if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]])
if not all_images:
    raise RuntimeError("❌ merged 이미지 없음 — Cell 2 먼저 실행하세요")

# ---- 클래스별 이미지 인덱스 사전 구축 ----
print("  클래스별 이미지 분류 중...")
cls_images = defaultdict(list)
for img_path in all_images:
    lbl = MERGED_LABELS / (img_path.stem + ".txt")
    if not lbl.exists():
        continue
    classes_in_img = set()
    for line in lbl.read_text(encoding="utf-8").strip().splitlines():
        parts = line.split()
        if parts:
            try: classes_in_img.add(int(parts[0]))
            except: pass
    for cid in classes_in_img:
        cls_images[cid].append(img_path)

# ---- 소수 클래스 최소 테스트셋 확보 ----
MIN_TEST_PER_CLASS = {
    2: 100,   # bollard  — 원래 32장 → 최소 100장 보장
    6: 100,   # crosswalk
}

random.seed(42)
forced_test = set()
for cid, min_n in MIN_TEST_PER_CLASS.items():
    available = cls_images.get(cid, [])
    random.shuffle(available)
    n_force = min(min_n, int(len(available) * 0.15))  # 최대 15%까지만
    forced = available[:n_force]
    forced_test.update(p.name for p in forced)
    print(f"  [{cid}] {CLASS_KO[cid]}: 테스트셋 최소 {n_force}장 확보")

# ---- 전체 분할 ----
remaining = [p for p in all_images if p.name not in forced_test]
forced_list = [p for p in all_images if p.name in forced_test]

random.seed(42)
random.shuffle(remaining)
n       = len(remaining)
n_train = int(n * 0.74)   # forced_test 빠진 만큼 train 비율 살짝 조정
n_val   = int(n * 0.21)

splits = {
    "train": remaining[:n_train],
    "val":   remaining[n_train:n_train + n_val],
    "test":  remaining[n_train + n_val:] + forced_list,  # forced는 무조건 test
}

split_counts = defaultdict(lambda: defaultdict(int))
for split_name, img_list in splits.items():
    for img_path in img_list:
        shutil.copy2(img_path, PROC_DIR / "images" / split_name / img_path.name)
        lbl = MERGED_LABELS / (img_path.stem + ".txt")
        if lbl.exists():
            shutil.copy2(lbl, PROC_DIR / "labels" / split_name / lbl.name)
            for line in lbl.read_text(encoding="utf-8").strip().splitlines():
                parts = line.split()
                if parts:
                    try: split_counts[split_name][int(parts[0])] += 1
                    except: pass
    print(f"  ✅ {split_name:5s}: {len(img_list):>6,}장")

# ---- 전체 YAML ----
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.dump({
        "path":  str(PROC_DIR).replace("\\", "/"),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    NUM_CLASSES,
        "names": CLASS_NAMES,
    }, f, default_flow_style=False, allow_unicode=True)
print(f"\n  📄 전체 dataset.yaml 저장: {CONFIG_PATH}")

# ---- 클래스별 서브셋 YAML ----
PER_CLASS_DATA_DIR = DATA_DIR / "per_class"
if PER_CLASS_DATA_DIR.exists():
    shutil.rmtree(PER_CLASS_DATA_DIR)

print("\n[클래스별 서브셋 생성 중...]")
CLASS_CONFIG_PATHS = {}

for target_cid, target_cname in enumerate(CLASS_NAMES):
    cls_dir = PER_CLASS_DATA_DIR / target_cname
    for split in ["train", "val", "test"]:
        (cls_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (cls_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    cnt = defaultdict(int)
    for split_name, img_list in splits.items():
        for img_path in img_list:
            lbl_src = PROC_DIR / "labels" / split_name / (img_path.stem + ".txt")
            if not lbl_src.exists():
                continue
            lines = lbl_src.read_text(encoding="utf-8").strip().splitlines()
            cls_lines = [l for l in lines if l.split() and int(l.split()[0]) == target_cid]
            if not cls_lines:
                continue
            img_src = PROC_DIR / "images" / split_name / img_path.name
            if not img_src.exists():
                continue
            shutil.copy2(img_src, cls_dir / "images" / split_name / img_path.name)
            (cls_dir / "labels" / split_name / lbl_src.name).write_text(
                "\n".join(cls_lines), encoding="utf-8")
            cnt[split_name] += 1

    cls_yaml_path = CONFIG_DIR / f"dataset_{target_cname}.yaml"
    with open(cls_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump({
            "path":  str(cls_dir).replace("\\", "/"),
            "train": "images/train",
            "val":   "images/val",
            "test":  "images/test",
            "nc":    NUM_CLASSES,
            "names": CLASS_NAMES,
        }, f, default_flow_style=False, allow_unicode=True)
    CLASS_CONFIG_PATHS[target_cid] = cls_yaml_path
    total_cls = sum(cnt.values())
    flag = "⚠️ 부족" if cnt.get("test", 0) < 50 else "✅"
    print(f"  [{target_cid}] {CLASS_KO[target_cid]:<8}: train={cnt['train']:,} val={cnt['val']:,} test={cnt['test']:,} → 총 {total_cls:,}장 {flag}")

# ---- 최종 분포 요약 ----
print("\n" + "=" * 60)
print(" 최종 클래스 분포")
print("=" * 60)
header = f"{'클래스':<12} {'Train':>8} {'Val':>8} {'Test':>8} {'합계':>8}"
print(header)
print("-" * len(header))
for cid, cname in enumerate(CLASS_NAMES):
    tr  = split_counts["train"].get(cid, 0)
    va  = split_counts["val"].get(cid, 0)
    te  = split_counts["test"].get(cid, 0)
    tot = tr + va + te
    flag = "❌" if tot == 0 else ("⚠️ " if te < 50 else "✅")
    print(f"[{cid}] {CLASS_KO[cid]:<10} {tr:>8,} {va:>8,} {te:>8,} {tot:>8,} {flag}")

## Cell 4 — Phase 1: 클래스별 개별 학습 (YOLOv8x)

In [ ]:
# ============================================================
# Cell 4: Phase 1 — 클래스별 개별 학습
# ============================================================
import torch

PER_CLASS_EPOCHS = {
    0: 40,  # bicycle
    1: 40,  # kickboard
    2: 25,  # bollard
    3: 15,  # car
    4: 30,  # motorcycle
    5: 30,  # stairs
    6: 20,  # crosswalk
    7: 40,  # people
}

PER_CLASS_AUG = {
    0: {"scale": 0.7, "translate": 0.15, "copy_paste": 0.4},
    1: {"scale": 0.7, "translate": 0.15, "copy_paste": 0.4},
    2: {"scale": 0.5, "translate": 0.1,  "copy_paste": 0.2},
    3: {"scale": 0.5, "translate": 0.1,  "copy_paste": 0.2},
    4: {"scale": 0.6, "translate": 0.12, "copy_paste": 0.3},
    5: {"scale": 0.5, "translate": 0.1,  "copy_paste": 0.2},
    6: {"scale": 0.5, "translate": 0.1,  "copy_paste": 0.2},
    7: {"scale": 0.7, "translate": 0.15, "copy_paste": 0.5, "degrees": 15.0},
}

BASE_CFG = {
    "device": DEVICE, "amp": True, "workers": 0,
    "batch": BATCH, "imgsz": 640,
    "patience": 10, "fraction": 1.0, "cache": False,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.01,
    "weight_decay": 0.0005, "warmup_epochs": 3, "cos_lr": True,
    "mosaic": 1.0, "mixup": 0.0, "close_mosaic": 10,
    "copy_paste": 0.2, "degrees": 10.0, "translate": 0.1,
    "scale": 0.5, "shear": 2.0, "perspective": 0.0,
    "flipud": 0.0, "fliplr": 0.5,
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "erasing": 0.3,
    "cls": 1.5, "box": 7.5, "dfl": 1.5,
    "save": True, "save_period": 10, "plots": True, "verbose": True,
}

print("=" * 60)
print(" Phase 1: 클래스별 개별 학습 (YOLOv8x)")
print("=" * 60)
print()
print("🎯 전략: 각 클래스를 전용 데이터로 집중 학습")
print("         → 클래스별 특징을 깊이 이해하도록 훈련")
print()

phase1_results = {}
TRAIN_ORDER = [7, 0, 1, 4, 5, 6, 2, 3]  # 어려운 순서대로

for cid in TRAIN_ORDER:
    cname    = CLASS_NAMES[cid]
    cname_ko = CLASS_KO[cid]
    cls_yaml = CLASS_CONFIG_PATHS.get(cid)
    if cls_yaml is None or not cls_yaml.exists():
        print(f"⏭️  [{cid}] {cname_ko}: YAML 없음, 스킵")
        continue

    save_dir = PHASE1_DIR / cname
    best_pt  = save_dir / "weights" / "best.pt"
    if best_pt.exists():
        print(f"⏭️  [{cid}] {cname_ko}: 이미 학습됨, 스킵")
        phase1_results[cid] = str(best_pt)
        continue

    epochs = PER_CLASS_EPOCHS[cid]
    print()
    print("=" * 55)
    print(f" [{cid}] {cname_ko} ({cname}) 학습 시작")
    print(f"       에폭: {epochs}  |  배치: {BATCH}  |  모델: YOLOv8x")
    print("=" * 55)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cfg = {**BASE_CFG}
    cfg.update(PER_CLASS_AUG.get(cid, {}))
    cfg["data"]     = str(cls_yaml)
    cfg["epochs"]   = epochs
    cfg["project"]  = str(PHASE1_DIR)
    cfg["name"]     = cname
    cfg["exist_ok"] = True

    try:
        model  = YOLO("yolov8x.pt")
        result = model.train(**cfg)
        phase1_results[cid] = str(best_pt)
        print(f"\n✅ [{cid}] {cname_ko} 학습 완료!")
        print(f"   저장 위치: {best_pt}")
    except Exception as e:
        print(f"\n❌ [{cid}] {cname_ko} 학습 실패: {e}")
        phase1_results[cid] = None

print()
print("=" * 60)
print(" Phase 1 완료 요약")
print("=" * 60)
for cid in range(NUM_CLASSES):
    pt_path = phase1_results.get(cid)
    status  = "✅ 성공" if pt_path and Path(pt_path).exists() else "❌ 실패"
    print(f"  [{cid}] {CLASS_KO[cid]:<8}: {status}")

## Cell 5 — Phase 1 중간 평가

In [ ]:
# ============================================================
# Cell 5: Phase 1 클래스별 모델 평가
# ============================================================
print("=" * 60)
print(" Phase 1 클래스별 모델 평가")
print("=" * 60)
print()
print("📊 각 클래스 전용 모델을 테스트셋으로 평가합니다.")
print("   (v7 결과와 비교)")
print()

V7_AP = {0: 0.622, 1: 0.542, 2: 0.742, 3: 0.798,
         4: 0.726, 5: 0.679, 6: 0.766, 7: 0.345}

phase1_ap = {}
print(f"{'클래스':<10} {'Phase1':>8} {'v7':>8} {'변화':>10} {'상태':>6}")
print("-" * 50)

for cid in range(NUM_CLASSES):
    cname    = CLASS_NAMES[cid]
    cname_ko = CLASS_KO[cid]
    best_pt  = PHASE1_DIR / cname / "weights" / "best.pt"
    if not best_pt.exists():
        print(f"[{cid}] {cname_ko:<8}: ❌ best.pt 없음")
        continue
    try:
        m        = YOLO(str(best_pt))
        cls_yaml = CLASS_CONFIG_PATHS.get(cid)
        if cls_yaml and cls_yaml.exists():
            metrics = m.val(data=str(cls_yaml), split="test",
                            device=DEVICE, batch=BATCH, workers=0, verbose=False)
            ap  = float(metrics.box.map50)
            phase1_ap[cid] = ap
            v7  = V7_AP.get(cid, 0)
            diff = ap - v7
            arrow = "↑" if diff > 0 else "↓"
            flag  = "✅" if ap >= 0.8 else ("⚠️ " if ap >= 0.6 else "❌")
            diff_str = f"{arrow}{abs(diff)*100:.1f}%p"
            print(f"[{cid}] {cname_ko:<8} {ap*100:>7.1f}% {v7*100:>7.1f}% {diff_str:>10} {flag}")
    except Exception as e:
        print(f"[{cid}] {cname_ko:<8}: 평가 실패 ({e})")

print()
print("→ Phase 2 (통합 파인튜닝)으로 이동하세요!")

## Cell 6 — Phase 2: 통합 파인튜닝 (YOLOv8x)

In [ ]:
# ============================================================
# Cell 6: Phase 2 — 통합 파인튜닝
# ============================================================
import torch

# ---- 초기 가중치 선택 ----
init_model_path = None

if phase1_ap:
    best_cid  = max(phase1_ap, key=phase1_ap.get)
    candidate = PHASE1_DIR / CLASS_NAMES[best_cid] / "weights" / "best.pt"
    if candidate.exists():
        init_model_path = str(candidate)
        print(f"✅ Phase 1 최고 모델: [{best_cid}] {CLASS_KO[best_cid]} (AP={phase1_ap[best_cid]*100:.1f}%)")

if not init_model_path:
    for try_cid in [7, 3, 0]:
        pt = PHASE1_DIR / CLASS_NAMES[try_cid] / "weights" / "best.pt"
        if pt.exists():
            init_model_path = str(pt)
            print(f"✅ 초기 가중치: {CLASS_KO[try_cid]} Phase1 모델")
            break

if not init_model_path:
    init_model_path = "yolov8x.pt"
    print("✅ 초기 가중치: yolov8x.pt (pretrained)")

PHASE2_CONFIG = {
    "data":          str(CONFIG_PATH),
    "device":        DEVICE,
    "amp":           True,
    "workers":       0,
    "epochs":        150,
    "batch":         BATCH,
    "imgsz":         640,
    "patience":      0,
    "fraction":      1.0,
    "cache":         False,
    "optimizer":     "AdamW",
    "lr0":           0.0005,
    "lrf":           0.005,
    "weight_decay":  0.0005,
    "warmup_epochs": 5,
    "cos_lr":        True,
    "mosaic":        1.0,
    "mixup":         0.05,
    "close_mosaic":  20,
    "copy_paste":    0.3,
    "degrees":       10.0,
    "translate":     0.1,
    "scale":         0.6,
    "shear":         2.0,
    "perspective":   0.0,
    "flipud":        0.0,
    "fliplr":        0.5,
    "hsv_h":         0.015,
    "hsv_s":         0.7,
    "hsv_v":         0.4,
    "erasing":       0.3,
    "cls":           2.0,
    "box":           7.5,
    "dfl":           1.5,
    "project":       str(MODELS_DIR),
    "name":          "safewalk_v8",
    "exist_ok":      True,
    "save":          True,
    "save_period":   10,
    "plots":         True,
    "verbose":       True,
}

print()
print("=" * 60)
print(" Phase 2: 통합 파인튜닝 (YOLOv8x)")
print("=" * 60)
print(f" 초기 모델  : {Path(init_model_path).name}")
print(f" 학습 에폭  : 150  (얼리스탑 없음)")
print(f" 학습률     : 0.0005 (파인튜닝)")
print(f" 배치 크기  : {BATCH}")
print(f" 이미지 크기: 640x640")
print(f" cls 손실   : 2.0 (클래스 구분 강화)")
print("=" * 60)
print()
print("💡 에폭별 결과 보는 법:")
print("   box_loss  → 낮을수록 좋음 (바운딩 박스 정확도)")
print("   cls_loss  → 낮을수록 좋음 (클래스 분류 정확도)")
print("   mAP50     → 높을수록 좋음 (탐지 성능 핵심 지표)")
print()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

RESUME    = False
last_ckpt = PHASE2_DIR / "weights" / "last.pt"

if RESUME and last_ckpt.exists():
    print(f"[이어서 학습] {last_ckpt}")
    model   = YOLO(str(last_ckpt))
    results = model.train(resume=True)
else:
    model   = YOLO(init_model_path)
    results = model.train(**PHASE2_CONFIG)

print()
print("✅ Phase 2 학습 완료!")
print(f"   최종 모델 저장: {PHASE2_DIR}/weights/best.pt")

## Cell 7 — 최종 평가 (테스트셋)

In [ ]:
# ============================================================
# Cell 7: 최종 평가
# ============================================================
best_pt = PHASE2_DIR / "weights" / "best.pt"
if not best_pt.exists():
    print(f"❌ {best_pt} 없음 — Cell 6 먼저 실행하세요")
else:
    print("평가 중... (약 1~2분 소요)")
    eval_model = YOLO(str(best_pt))
    metrics    = eval_model.val(
        data=str(CONFIG_PATH), split="test",
        device=DEVICE, batch=BATCH, workers=0, verbose=True,
    )

    V7_AP  = [0.622, 0.542, 0.742, 0.798, 0.726, 0.679, 0.766, 0.345]
    V7_MAP = 0.652

    map50    = metrics.box.map50
    map5095  = metrics.box.map
    prec     = metrics.box.mp
    rec      = metrics.box.mr

    print()
    print("=" * 65)
    print(" 🎯 SafeWalk v8 최종 평가 결과")
    print("=" * 65)
    print()
    print("📌 전체 성능 요약")
    print("-" * 50)
    print(f"  mAP@0.5       : {map50*100:.1f}%  (v7: {V7_MAP*100:.1f}%  변화: {(map50-V7_MAP)*100:+.1f}%p)")
    print(f"  mAP@0.5:0.95  : {map5095*100:.1f}%")
    print(f"  Precision     : {prec*100:.1f}%")
    print(f"  Recall        : {rec*100:.1f}%")
    print()
    print("💡 용어 설명")
    print("  mAP@0.5      → 전체 탐지 정확도. 높을수록 좋음. 80% 이상이면 우수")
    print("  Precision    → '탐지했을 때 맞을 확률'. 오탐지가 적을수록 높음")
    print("  Recall       → '실제 객체를 찾아낸 비율'. 놓치는 게 적을수록 높음")
    print()
    print("=" * 65)
    print(" 📊 클래스별 상세 결과 (v7 대비 비교)")
    print("=" * 65)
    print(f"  {'클래스':<10} {'v8 결과':>8} {'v7 결과':>8} {'변화':>10} {'상태':>6} {'평가'}")
    print("  " + "-" * 62)

    if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
        for cid, cname_ko in enumerate(CLASS_KO):
            if cid >= len(metrics.box.ap50):
                continue
            ap   = float(metrics.box.ap50[cid])
            v7   = V7_AP[cid]
            diff = ap - v7
            arrow = "↑" if diff > 0 else "↓"
            flag  = "✅" if ap >= 0.8 else ("⚠️ " if ap >= 0.6 else "❌")
            diff_str = f"{arrow}{abs(diff)*100:.1f}%p"
            if ap >= 0.8:   comment = "우수"
            elif ap >= 0.6: comment = "양호"
            elif ap >= 0.4: comment = "개선 필요"
            else:            comment = "데이터 추가 필요"
            print(f"  [{cid}] {cname_ko:<8} {ap*100:>7.1f}% {v7*100:>7.1f}% {diff_str:>10} {flag}  {comment}")

    print()
    print("=" * 65)
    print(" 📈 전체 평가 요약")
    print("=" * 65)

    if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
        ap_vals  = [float(metrics.box.ap50[i]) for i in range(min(8, len(metrics.box.ap50)))]
        above_80 = sum(1 for v in ap_vals if v >= 0.8)
        above_60 = sum(1 for v in ap_vals if 0.6 <= v < 0.8)
        below_60 = sum(1 for v in ap_vals if v < 0.6)
        print(f"  80% 이상 (우수) : {above_80}개 클래스")
        print(f"  60~80% (양호)   : {above_60}개 클래스")
        print(f"  60% 미만 (부족) : {below_60}개 클래스")
        print()
        print(f"  최고 성능 클래스: {CLASS_KO[int(np.argmax(ap_vals))]} ({max(ap_vals)*100:.1f}%)")
        print(f"  최저 성능 클래스: {CLASS_KO[int(np.argmin(ap_vals))]} ({min(ap_vals)*100:.1f}%)")
        print(f"  평균 AP@0.5    : {np.mean(ap_vals)*100:.1f}%")
        if below_60 == 0:
            print()
            print("  🎉 모든 클래스가 60% 이상을 달성했습니다!")
        elif above_80 >= 6:
            print()
            print("  👍 대부분의 클래스가 80% 이상을 달성했습니다!")

## Cell 8 — 실시간 처리 최적화 (TensorRT Export)

In [ ]:
# ============================================================
# Cell 8: 실시간 처리 최적화
# ============================================================
import time
import numpy as np

best_pt = PHASE2_DIR / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(f"❌ {best_pt} 없음 — Cell 6 먼저 실행하세요")

print("=" * 60)
print(" 실시간 처리 최적화")
print("=" * 60)

trt_path = PHASE2_DIR / "weights" / "best.engine"

# ---- TensorRT Export ----
if DEVICE != "cpu" and torch.cuda.is_available():
    if not trt_path.exists():
        print("\n[1] TensorRT 엔진 변환 중... (3~5분 소요)")
        print("    → GPU에서 최고 속도로 실행하기 위한 최적화 과정입니다")
        try:
            export_model = YOLO(str(best_pt))
            export_model.export(
                format="engine", imgsz=640, half=True,
                device=0, workspace=4, verbose=False,
            )
            print(f"    ✅ TensorRT 엔진 저장 완료: {trt_path}")
        except Exception as e:
            print(f"    ⚠️ TensorRT 변환 실패: {e}")
    else:
        print(f"\n[1] TensorRT 엔진 이미 존재: {trt_path}")
else:
    print("\n[1] CPU 모드 — TensorRT 스킵")


def benchmark_fps(model_path, n_warmup=10, n_test=100):
    model = YOLO(str(model_path))
    dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    for _ in range(n_warmup):
        model.predict(dummy, verbose=False, device=DEVICE)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_test):
        model.predict(dummy, verbose=False, device=DEVICE)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return n_test / elapsed, elapsed / n_test * 1000


print("\n[2] 추론 속도 측정")
print("    (100장 연속 추론 후 평균 계산)")
print()

try:
    fps, ms = benchmark_fps(best_pt)
    print(f"  PyTorch FP32 : {fps:.0f} FPS  ({ms:.1f} ms/프레임)")
    print(f"  {'✅ 실시간 가능!' if fps >= 30 else '⚠️ 실시간 어려움'}")
except Exception as e:
    print(f"  PyTorch 측정 실패: {e}")

if trt_path.exists():
    try:
        fps_trt, ms_trt = benchmark_fps(trt_path)
        print(f"\n  TensorRT FP16: {fps_trt:.0f} FPS  ({ms_trt:.1f} ms/프레임)")
        print(f"  {'✅ 실시간 가능!' if fps_trt >= 30 else '⚠️ 실시간 어려움'}")
        print(f"  속도 향상: {fps_trt/fps:.1f}배 빠름")
    except Exception as e:
        print(f"\n  TensorRT 측정 실패: {e}")

print()
print("[3] 실시간 추론 설정 저장")
REALTIME_CONFIG = {
    "conf": 0.4, "iou": 0.45, "imgsz": 640,
    "half": True if DEVICE != "cpu" else False,
    "device": DEVICE, "max_det": 50,
    "verbose": False, "stream": True,
}
rt_config_path = PHASE2_DIR / "realtime_config.json"
with open(rt_config_path, "w") as f:
    json.dump(REALTIME_CONFIG, f, indent=2)
print(f"  ✅ 설정 저장: {rt_config_path}")
print()
print("💡 설정 설명:")
print(f"  conf=0.4  → 40% 미만 확신이면 탐지 제외")
print(f"  iou=0.45  → 박스가 45% 이상 겹치면 하나로 합침")
print(f"  half=True → FP16 사용 (속도 2배, 메모리 절반)")
print(f"  max_det=50→ 한 프레임 최대 50개까지만 탐지")

## Cell 9 — 실시간 추론 데모

In [ ]:
# ============================================================
# Cell 9: 실시간 추론 데모
# ============================================================
import cv2, time, json
import numpy as np
from pathlib import Path
from ultralytics import YOLO

trt_path   = PHASE2_DIR / "weights" / "best.engine"
best_pt    = PHASE2_DIR / "weights" / "best.pt"
MODEL_PATH = str(trt_path) if trt_path.exists() else str(best_pt)
print(f"사용 모델: {Path(MODEL_PATH).name}")

rt_config_path = PHASE2_DIR / "realtime_config.json"
RT = json.load(open(rt_config_path)) if rt_config_path.exists() else {
    "conf": 0.4, "iou": 0.45, "imgsz": 640,
    "half": False, "device": DEVICE, "max_det": 50, "verbose": False
}

CLASS_COLORS = [
    (0,   165, 255),  # 자전거   - 주황
    (255, 0,   255),  # 킥보드   - 마젠타
    (0,   255, 255),  # 볼라드   - 노랑
    (0,   0,   255),  # 자동차   - 빨강
    (255, 128, 0  ),  # 오토바이 - 파랑주황
    (128, 0,   255),  # 계단     - 보라
    (0,   255, 0  ),  # 횡단보도 - 초록
    (255, 255, 0  ),  # 보행자   - 하늘
]


class SafeWalkDetector:
    def __init__(self, model_path, rt_config):
        self.model   = YOLO(model_path)
        self.rt      = rt_config
        self.fps_buf = []
        self.prev_t  = time.perf_counter()

    def predict_frame(self, frame):
        return self.model.predict(
            frame,
            conf=self.rt.get("conf", 0.4),
            iou=self.rt.get("iou", 0.45),
            imgsz=self.rt.get("imgsz", 640),
            half=self.rt.get("half", False),
            device=self.rt.get("device", "cpu"),
            max_det=self.rt.get("max_det", 50),
            verbose=False,
        )[0]

    def draw_results(self, frame, result):
        h, w = frame.shape[:2]
        now  = time.perf_counter()
        fps  = 1.0 / max(now - self.prev_t, 1e-6)
        self.prev_t = now
        self.fps_buf.append(fps)
        if len(self.fps_buf) > 30:
            self.fps_buf.pop(0)
        avg_fps = np.mean(self.fps_buf)

        detected = []
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                cid  = int(box.cls[0])
                conf = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                color = CLASS_COLORS[cid] if cid < len(CLASS_COLORS) else (255,255,255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                label = f"{CLASS_KO[cid]} {conf:.0%}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
                cv2.putText(frame, label, (x1+3, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
                detected.append((cid, conf))

        cv2.rectangle(frame, (0, 0), (280, 55), (0,0,0), -1)
        cv2.putText(frame, f"SafeWalk v8 | FPS: {avg_fps:.0f}",
                    (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 1)
        cv2.putText(frame, f"탐지: {len(detected)}개 객체",
                    (8, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

        danger_cls = [cid for cid, _ in detected if cid in [0,1,3,4]]
        if danger_cls:
            alert = f"주의: {', '.join(CLASS_KO[c] for c in set(danger_cls))}"
            cv2.rectangle(frame, (w-320, 0), (w, 35), (0,0,180), -1)
            cv2.putText(frame, alert, (w-315, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)

        return frame, avg_fps, detected

    def run_webcam(self, cam_id=0, save_path=None):
        cap = cv2.VideoCapture(cam_id)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        cap.set(cv2.CAP_PROP_FPS, 30)
        writer = None
        if save_path:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(save_path, fourcc, 30, (1280, 720))
        print("웹캠 시작 — 'q' 키로 종료")
        while True:
            ret, frame = cap.read()
            if not ret: break
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            if writer: writer.write(frame)
            cv2.imshow("SafeWalk v8 Real-time", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
        cap.release()
        if writer: writer.release()
        cv2.destroyAllWindows()

    def run_video(self, video_path, save_path=None, show=False):
        cap   = cv2.VideoCapture(str(video_path))
        fps_o = cap.get(cv2.CAP_PROP_FPS) or 30
        W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        writer = None
        if save_path:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(save_path), fourcc, fps_o, (W, H))
        print(f"동영상 추론: {video_path} ({total}프레임)")
        fps_list = []
        for i in range(total):
            ret, frame = cap.read()
            if not ret: break
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            fps_list.append(fps)
            if writer: writer.write(frame)
            if show:
                cv2.imshow("SafeWalk v8", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'): break
            if (i+1) % 100 == 0:
                print(f"  {i+1}/{total} | 평균 FPS: {np.mean(fps_list):.1f}")
        cap.release()
        if writer: writer.release()
        if show: cv2.destroyAllWindows()
        avg = np.mean(fps_list) if fps_list else 0
        print(f"완료 | 평균 FPS: {avg:.1f}")
        if save_path: print(f"저장: {save_path}")

    def run_image_folder(self, img_dir, save_dir=None):
        imgs = list(Path(img_dir).glob("*.jpg")) + list(Path(img_dir).glob("*.png"))
        if save_dir: Path(save_dir).mkdir(parents=True, exist_ok=True)
        print(f"이미지 추론: {len(imgs)}장")
        for i, img_path in enumerate(imgs):
            frame  = cv2.imread(str(img_path))
            result = self.predict_frame(frame)
            frame, fps, det = self.draw_results(frame, result)
            if save_dir:
                cv2.imwrite(str(Path(save_dir) / img_path.name), frame)
            if (i+1) % 10 == 0:
                print(f"  {i+1}/{len(imgs)} 완료")
        print("완료")


detector = SafeWalkDetector(MODEL_PATH, RT)
print("\n✅ SafeWalkDetector 준비 완료!")
print()
print("사용법:")
print("  detector.run_webcam(cam_id=0)                              # 웹캠")
print("  detector.run_video('input.mp4', save_path='output.mp4')   # 동영상")
print("  detector.run_image_folder('/images', save_dir='/output')   # 이미지 폴더")

## Cell 10 — 결과 시각화 (한글 지원)

In [ ]:
# ============================================================
# Cell 10: 시각화 — 한글 완전 지원
# ============================================================
import subprocess
subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], check=False)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import seaborn as sns

# ---- 한글 폰트 설정 ----
fm._load_fontmanager(try_read_cache=False)
nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f or 'nanum' in f]
if nanum:
    font_prop = fm.FontProperties(fname=nanum[0])
    matplotlib.rc('font', family=font_prop.get_name())
    print(f'✅ 한글 폰트: {font_prop.get_name()}')
else:
    matplotlib.rc('font', family='DejaVu Sans')
    print('⚠️ 나눔폰트 없음 — 한글 깨질 수 있음')
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 130

RESULTS_DIR = PHASE2_DIR
CSV_PATH    = RESULTS_DIR / 'results.csv'
SAVE_DIR    = RESULTS_DIR / 'plots'
SAVE_DIR.mkdir(exist_ok=True)

if not CSV_PATH.exists():
    print(f'❌ {CSV_PATH} 없음 — Cell 6 완료 후 실행하세요')
else:
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    print(f'✅ 학습 로그 로드: {len(df)} 에폭')

    epochs = df['epoch'].values if 'epoch' in df.columns else np.arange(len(df))

    COL = {}
    col_map = {
        'train_box': ['train/box_loss', 'train/box_om'],
        'train_cls': ['train/cls_loss', 'train/cls_om'],
        'train_dfl': ['train/dfl_loss', 'train/dfl_om'],
        'val_box':   ['val/box_loss',   'val/box_om'],
        'val_cls':   ['val/cls_loss',   'val/cls_om'],
        'val_dfl':   ['val/dfl_loss',   'val/dfl_om'],
        'precision': ['metrics/precision(B)', 'metrics/precision'],
        'recall':    ['metrics/recall(B)',    'metrics/recall'],
        'map50':     ['metrics/mAP50(B)',     'metrics/mAP_0.5'],
        'map50_95':  ['metrics/mAP50-95(B)',  'metrics/mAP_0.5:0.95'],
        'lr':        ['lr/pg0', 'lr0'],
    }
    for key, candidates in col_map.items():
        for c in candidates:
            if c in df.columns:
                COL[key] = c
                break

    V6_AP = [0.480, 0.445, 0.737, 0.858, 0.656, 0.708, 0.736, 0.329]
    V7_AP = [0.622, 0.542, 0.742, 0.798, 0.726, 0.679, 0.766, 0.345]

    # ---- 최종 AP 계산 ----
    ap50_vals = None
    best_pt_path = RESULTS_DIR / 'weights' / 'best.pt'
    if best_pt_path.exists():
        try:
            _m   = YOLO(str(best_pt_path))
            _met = _m.val(data=str(CONFIG_PATH), split='test',
                          device=DEVICE, batch=BATCH, workers=0, verbose=False)
            if hasattr(_met.box, 'ap50') and _met.box.ap50 is not None:
                ap50_vals = [float(v) for v in _met.box.ap50[:8]]
        except Exception as e:
            print(f'AP 재계산 실패: {e}')

    C = {
        'primary':  '#2563EB',
        'success':  '#16A34A',
        'warning':  '#D97706',
        'danger':   '#DC2626',
        'purple':   '#7C3AED',
        'pink':     '#DB2777',
        'bg':       '#F8FAFC',
        'grid':     '#E2E8F0',
        'text':     '#1E293B',
        'subtext':  '#64748B',
    }

    def make_ax(ax, title, xlabel='에폭', ylabel=''):
        ax.set_title(title, fontsize=12, fontweight='bold', pad=10, color=C['text'])
        ax.set_xlabel(xlabel, fontsize=9, color=C['subtext'])
        ax.set_ylabel(ylabel, fontsize=9, color=C['subtext'])
        ax.grid(True, alpha=0.4, linestyle='--', color=C['grid'])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color(C['grid'])
        ax.spines['bottom'].set_color(C['grid'])
        ax.set_facecolor(C['bg'])
        ax.tick_params(colors=C['subtext'], labelsize=8)

    def best_marker(ax, epochs, vals, color='red'):
        idx  = vals.idxmax()
        bep  = epochs[idx]
        bval = vals.max()
        ax.axvline(bep, color=color, linestyle=':', alpha=0.7, lw=1.5)
        ax.annotate(
            f'최고\n{bval*100:.1f}%\n(ep {int(bep)})',
            xy=(bep, bval),
            xytext=(bep + len(epochs) * 0.08, bval - 0.08),
            fontsize=8, color=color,
            arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

    fig = plt.figure(figsize=(28, 38))
    fig.patch.set_facecolor('white')
    fig.suptitle(
        'SafeWalk v8  학습 결과 종합 리포트\n'
        'YOLOv8x  |  Phase1 클래스별 + Phase2 통합 파인튜닝',
        fontsize=22, fontweight='bold', y=0.995, color=C['text']
    )

    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ① Train Loss 3종
    ax1 = fig.add_subplot(gs[0, 0])
    for k, lbl, c in [('train_box','박스 손실',C['primary']),
                       ('train_cls','클래스 손실',C['warning']),
                       ('train_dfl','DFL 손실',C['success'])]:
        if k in COL:
            ax1.plot(epochs, df[COL[k]], color=c, lw=2, label=lbl)
    make_ax(ax1, '① 학습 손실 3종 (낮을수록 좋음)', ylabel='손실값')
    ax1.legend(fontsize=8, loc='upper right')

    # ② Val Loss 3종
    ax2 = fig.add_subplot(gs[0, 1])
    for k, lbl, c in [('val_box','박스 손실',C['primary']),
                       ('val_cls','클래스 손실',C['warning']),
                       ('val_dfl','DFL 손실',C['success'])]:
        if k in COL:
            ax2.plot(epochs, df[COL[k]], color=c, lw=2, label=lbl)
    make_ax(ax2, '② 검증 손실 3종 (낮을수록 좋음)', ylabel='손실값')
    ax2.legend(fontsize=8, loc='upper right')

    # ③ Train vs Val Box Loss
    ax3 = fig.add_subplot(gs[0, 2])
    if 'train_box' in COL:
        ax3.plot(epochs, df[COL['train_box']], C['primary'], lw=2, label='학습')
    if 'val_box' in COL:
        ax3.plot(epochs, df[COL['val_box']], C['danger'], lw=2, ls='--', label='검증')
        if 'train_box' in COL:
            ax3.fill_between(epochs, df[COL['train_box']], df[COL['val_box']],
                             where=(df[COL['val_box']] > df[COL['train_box']]),
                             alpha=0.1, color=C['danger'], label='과적합 구간')
    make_ax(ax3, '③ 학습 vs 검증 박스 손실\n(두 선이 벌어지면 과적합)', ylabel='손실값')
    ax3.legend(fontsize=8)

    # ④ mAP@0.5
    ax4 = fig.add_subplot(gs[1, 0])
    if 'map50' in COL:
        vals = df[COL['map50']]
        ax4.plot(epochs, vals, C['primary'], lw=2.5, label='mAP@0.5')
        ax4.fill_between(epochs, vals, alpha=0.12, color=C['primary'])
        best_marker(ax4, epochs, vals, color=C['danger'])
        ax4.axhline(0.8, color=C['success'], ls='--', alpha=0.5, lw=1.5)
        ax4.text(epochs[0]+1, 0.81, '목표 80%', fontsize=8, color=C['success'], alpha=0.8)
    make_ax(ax4, '④ mAP@0.5 추이\n(전체 탐지 정확도, 높을수록 좋음)', ylabel='mAP@0.5')
    ax4.set_ylim(0, 1.05)

    # ⑤ mAP@0.5:0.95
    ax5 = fig.add_subplot(gs[1, 1])
    if 'map50_95' in COL:
        vals = df[COL['map50_95']]
        ax5.plot(epochs, vals, C['purple'], lw=2.5)
        ax5.fill_between(epochs, vals, alpha=0.12, color=C['purple'])
        best_marker(ax5, epochs, vals, color=C['purple'])
    make_ax(ax5, '⑤ mAP@0.5:0.95 추이\n(박스 위치까지 포함한 정밀 정확도)', ylabel='mAP@0.5:0.95')
    ax5.set_ylim(0, 1.05)

    # ⑥ Precision & Recall
    ax6 = fig.add_subplot(gs[1, 2])
    if 'precision' in COL:
        ax6.plot(epochs, df[COL['precision']], C['success'], lw=2, label='Precision (정밀도)')
    if 'recall' in COL:
        ax6.plot(epochs, df[COL['recall']], C['danger'], lw=2, label='Recall (재현율)')
    make_ax(ax6, '⑥ Precision & Recall\nPrecision=오탐↓  Recall=놓침↓', ylabel='점수')
    ax6.set_ylim(0, 1.05)
    ax6.legend(fontsize=8)

    # ⑦ v6 vs v7 vs v8 비교
    ax7 = fig.add_subplot(gs[2, 0:2])
    x = np.arange(len(CLASS_KO))
    w = 0.25
    ax7.bar(x - w, V6_AP, w, label='v6', color='#94A3B8', alpha=0.8, edgecolor='white')
    ax7.bar(x,     V7_AP, w, label='v7', color=C['warning'], alpha=0.8, edgecolor='white')
    if ap50_vals:
        bar_colors = [C['success'] if v >= 0.8 else C['warning'] if v >= 0.6 else C['danger']
                      for v in ap50_vals]
        bars = ax7.bar(x + w, ap50_vals, w, label='v8', color=bar_colors, alpha=0.9, edgecolor='white')
        for bar, val in zip(bars, ap50_vals):
            ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val*100:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax7.axhline(0.8, color=C['success'], ls='--', alpha=0.5, lw=1.5, label='목표 80%')
    ax7.set_xticks(x)
    ax7.set_xticklabels(CLASS_KO, fontsize=10)
    ax7.set_ylim(0, 1.15)
    ax7.legend(fontsize=9, loc='upper right')
    make_ax(ax7, '⑦ v6 → v7 → v8 클래스별 AP@0.5 비교', xlabel='', ylabel='AP@0.5')

    # ⑧ LR 스케줄
    ax8 = fig.add_subplot(gs[2, 2])
    if 'lr' in COL:
        ax8.plot(epochs, df[COL['lr']], '#92400E', lw=2)
        ax8.fill_between(epochs, df[COL['lr']], alpha=0.15, color='#92400E')
    make_ax(ax8, '⑧ 학습률 스케줄\n(코사인 감소)', ylabel='학습률')

    # ⑨ 학습 요약 테이블
    ax9 = fig.add_subplot(gs[3, 0])
    ax9.axis('off')
    rows = []
    if 'map50'    in COL: rows.append(['최고 mAP@0.5',      f"{df[COL['map50']].max()*100:.2f}%"])
    if 'map50'    in COL: rows.append(['최고 에폭',          str(int(epochs[df[COL['map50']].idxmax()]))])
    if 'map50_95' in COL: rows.append(['최고 mAP@0.5:0.95', f"{df[COL['map50_95']].max()*100:.2f}%"])
    if 'precision'in COL: rows.append(['최고 Precision',    f"{df[COL['precision']].max()*100:.2f}%"])
    if 'recall'   in COL: rows.append(['최고 Recall',       f"{df[COL['recall']].max()*100:.2f}%"])
    rows.append(['모델', 'YOLOv8x'])
    rows.append(['총 에폭', str(len(df))])
    if ap50_vals:
        rows.append(['최고 클래스', f"{CLASS_KO[int(np.argmax(ap50_vals))]} ({max(ap50_vals)*100:.1f}%)"])
        rows.append(['최저 클래스', f"{CLASS_KO[int(np.argmin(ap50_vals))]} ({min(ap50_vals)*100:.1f}%)"])
        v8_mean = np.mean(ap50_vals)
        v6_mean = np.mean(V6_AP)
        rows.append(['v6→v8 평균 개선', f"{(v8_mean-v6_mean)*100:+.1f}%p"])
    tbl = ax9.table(cellText=rows, colLabels=['항목', '값'],
                    cellLoc='center', loc='center', colWidths=[0.6, 0.4])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.8)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(C['primary'])
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#EFF6FF')
        cell.set_edgecolor('#CBD5E1')
    ax9.set_title('⑨ 학습 요약', fontsize=12, fontweight='bold', pad=10, color=C['text'])

    # ⑩ v8 클래스별 AP 가로 막대
    ax10 = fig.add_subplot(gs[3, 1])
    if ap50_vals:
        bar_colors = [C['success'] if v >= 0.8 else C['warning'] if v >= 0.6 else C['danger']
                      for v in ap50_vals]
        bars = ax10.barh(CLASS_KO[::-1], ap50_vals[::-1], color=bar_colors[::-1],
                         edgecolor='white', linewidth=1.5, height=0.6)
        for bar, val in zip(bars, ap50_vals[::-1]):
            ax10.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                      f'{val*100:.1f}%', va='center', fontsize=9, fontweight='bold')
        ax10.axvline(0.8, color=C['success'], ls='--', alpha=0.6, lw=1.5)
        ax10.axvline(0.6, color=C['warning'], ls='--', alpha=0.4, lw=1)
        ax10.set_xlim(0, 1.15)
        legend_patches = [
            mpatches.Patch(color=C['success'], label='80% 이상 ✅ 우수'),
            mpatches.Patch(color=C['warning'], label='60~80% ⚠️ 양호'),
            mpatches.Patch(color=C['danger'],  label='60% 미만 ❌ 개선 필요'),
        ]
        ax10.legend(handles=legend_patches, fontsize=8, loc='lower right')
    make_ax(ax10, '⑩ v8 클래스별 AP@0.5', xlabel='AP@0.5', ylabel='')

    # ⑪ 과적합 진단
    ax11 = fig.add_subplot(gs[3, 2])
    if all(k in COL for k in ['train_box', 'train_cls', 'val_box', 'val_cls']):
        train_total = df[COL['train_box']] + df[COL['train_cls']]
        val_total   = df[COL['val_box']]   + df[COL['val_cls']]
        ax11.plot(epochs, train_total, C['primary'], lw=2, label='학습 손실 합계')
        ax11.plot(epochs, val_total,   C['danger'],  lw=2, ls='--', label='검증 손실 합계')
        ax11.fill_between(epochs, train_total, val_total,
                          where=(val_total > train_total),
                          alpha=0.15, color=C['danger'], label='과적합 구간')
        ax11.legend(fontsize=8)
    make_ax(ax11, '⑪ 과적합 진단\n(두 선 간격이 클수록 과적합 심함)', ylabel='손실 합계')

    plt.tight_layout(rect=[0, 0, 1, 0.985])
    save_path = SAVE_DIR / 'safewalk_v8_결과.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'\n✅ 그래프 저장: {save_path}')